# LoRA GPT-2 Medium E2E Training + Evaluation on Google Colab

Use this notebook to rerun the GPT-2 Medium + LoRA + E2E NLG reproduction on a Colab GPU.

This combined notebook:

- clones or updates the project,
- downloads and preprocesses the official Microsoft LoRA E2E files,
- runs tests and dry-run checks,
- trains LoRA adapters with validation loss/perplexity logged at the end of each epoch,
- generates full E2E test predictions,
- computes grouped multi-reference E2E metrics,
- creates report figures,
- backs up outputs to Google Drive.

Recommended runtime: `Runtime > Change runtime type > GPU`. A T4 should work; L4/A100 will be faster.

## 1. Check GPU

Run this first to confirm Colab assigned a CUDA GPU.

In [1]:
!nvidia-smi

import torch
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

Sat Apr 25 23:08:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P0             49W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 2. Clone Or Update The Repo

If the repo is private, paste a GitHub token. If it is public for your session, press Enter.

In [2]:
from getpass import getpass
from pathlib import Path
import os
import subprocess

REPO_OWNER = "justinlxiang"
REPO_NAME = "CS4782-final-project"
BRANCH = "main"
PROJECT_DIR = Path("/content") / REPO_NAME
WORK_DIR = PROJECT_DIR / "lora-gpt2-medium-e2e"

token = getpass("GitHub token, or press Enter for public clone: ")
repo_url = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"
if token:
    repo_url = f"https://{token}@github.com/{REPO_OWNER}/{REPO_NAME}.git"

if PROJECT_DIR.exists():
    subprocess.run(["git", "-C", str(PROJECT_DIR), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, repo_url, str(PROJECT_DIR)], check=True)

os.chdir(WORK_DIR)
print("working directory:", Path.cwd())
!git log --oneline -3

GitHub token, or press Enter for public clone: ··········
working directory: /content/CS4782-final-project/lora-gpt2-medium-e2e
2efa46b (HEAD -> main, origin/main, origin/HEAD) remove figures dir
f7e6166 Merge Colab training and evaluation workflow
ff08772 Add official-style beam decoding


## 3. Install Dependencies

In [3]:
!pip install -q -r requirements.txt

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.7 MB/s eta 0:00:00


## 4. Mount Google Drive

Backups are written under one run folder in Drive:

```text
/content/drive/MyDrive/e2e_lora_r4_alpha32/
```

The run folder mirrors `outputs/runs/e2e_lora_r4_alpha32/` and includes checkpoints, metrics, generations, grouped E2E files, config, notebook, and `figures/`.

In [4]:
from google.colab import drive
from pathlib import Path
import shutil

# Mount Drive once, then keep everything for this experiment inside one run folder.
drive.mount('/content/drive')

DRIVE_RUN_DIR = Path('/content/drive/MyDrive/e2e_lora_r4_alpha32')
LOCAL_RUN_DIR = Path('outputs/runs/e2e_lora_r4_alpha32')
LOCAL_ADAPTER = LOCAL_RUN_DIR / 'checkpoints' / 'adapter_final.pt'
DRIVE_RUN_DIR.mkdir(parents=True, exist_ok=True)


def backup_to_run(relative_path: str | Path, destination_name: str | None = None) -> Path | None:
    """Copy a file/folder into the Drive run folder."""
    source = Path(relative_path)
    if not source.exists():
        print('skip missing:', source)
        return None
    destination = DRIVE_RUN_DIR / (destination_name or source.name)
    if source.is_dir():
        shutil.copytree(source, destination, dirs_exist_ok=True)
    else:
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, destination)
    print('backed up:', source, '->', destination)
    return destination

print('Drive run dir:', DRIVE_RUN_DIR)
print('Local run dir:', LOCAL_RUN_DIR)

Mounted at /content/drive
Drive run dir: /content/drive/MyDrive/e2e_lora_r4_alpha32
Local run dir: outputs/runs/e2e_lora_r4_alpha32


## 5. Download E2E Dataset

These are the official E2E files used by the Microsoft LoRA NLG example. Each raw row has `context||completion`.

In [5]:
!mkdir -p data/raw/e2e
!curl -L -o data/raw/e2e/train.txt https://raw.githubusercontent.com/microsoft/LoRA/main/examples/NLG/data/e2e/train.txt
!curl -L -o data/raw/e2e/valid.txt https://raw.githubusercontent.com/microsoft/LoRA/main/examples/NLG/data/e2e/valid.txt
!curl -L -o data/raw/e2e/test.txt https://raw.githubusercontent.com/microsoft/LoRA/main/examples/NLG/data/e2e/test.txt
!wc -l data/raw/e2e/*.txt

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 9398k  100 9398k    0     0  11.1M      0 --:--:-- --:--:-- --:--:-- 11.1M
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 1170k  100 1170k    0     0  2530k      0 --:--:-- --:--:-- --:--:-- 2532k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 1319k  100 1319k    0     0  2304k      0 --:--:-- --:--:-- --:--:-- 2302k
    4693 data/raw/e2e/test.txt
   42061 data/raw/e2e/train.txt
    4672 data/raw/e2e/valid.txt
   51426 total


## 6. Preprocess E2E

This creates `data/processed/e2e_gpt2/*.jsonl` with the official-style token sequence:

`raw_context + 50256 + leading_space_completion + 50256`

In [6]:
!python scripts/prepare_e2e.py --config configs/e2e_gpt2_medium_lora.yaml

import json
from pathlib import Path

example = json.loads(Path('data/processed/e2e_gpt2/train.jsonl').read_text().splitlines()[0])
print('prompt:', example['prompt'])
print('first input ids:', example['input_ids'][:20])
print('first labels:', example['labels'][:20])
print('prompt length:', example['prompt_length'])

config.json: 100% 718/718 [00:00<00:00, 3.00MB/s]
tokenizer_config.json: 100% 26.0/26.0 [00:00<00:00, 147kB/s]
vocab.json: 1.04MB [00:00, 12.6MB/s]
merges.txt: 456kB [00:00, 10.9MB/s]
tokenizer.json: 1.36MB [00:00, 19.8MB/s]
wrote 42061 examples to /content/CS4782-final-project/lora-gpt2-medium-e2e/data/processed/e2e_gpt2/train.jsonl
wrote 4672 examples to /content/CS4782-final-project/lora-gpt2-medium-e2e/data/processed/e2e_gpt2/valid.jsonl
wrote 4693 examples to /content/CS4782-final-project/lora-gpt2-medium-e2e/data/processed/e2e_gpt2/test.jsonl
prompt: name : The Vaults | Type : pub | price : more than £ 30 | customer rating : 5 out of 5 | near : Café Adriatic
first input ids: [3672, 1058, 383, 21314, 930, 5994, 1058, 2240, 930, 2756, 1058, 517, 621, 4248, 1542, 930, 6491, 7955, 1058, 642]
first labels: [-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100]
prompt length: 31


## 7. Run Tests And Dry Run

This verifies LoRA injection, preprocessing, checkpoint helpers, generation helpers, evaluation grouping, and one forward loss pass.

In [7]:
!python -m pytest
!python scripts/count_params.py --config configs/e2e_gpt2_medium_lora.yaml
!python scripts/train.py --config configs/e2e_gpt2_medium_lora.yaml --dry-run --device cuda --dry-run-forward-pass

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0
rootdir: /content/CS4782-final-project/lora-gpt2-medium-e2e
configfile: pyproject.toml
testpaths: tests
plugins: langsmith-0.7.30, typeguard-4.5.1, anyio-4.13.0
collected 24 items                                                             

tests/test_checkpointing.py ...                                          [ 12%]
tests/test_data.py .....                                                 [ 33%]
tests/test_evaluate.py ....                                              [ 50%]
tests/test_generation.py .....                                           [ 70%]
tests/test_inject.py .                                                   [ 75%]
tests/test_lora_layers.py ...                                            [ 87%]
tests/test_train.py .                                                    [ 91%]
tests/test_utils.py ..                                  

## 8. Optional: Short Smoke Training

This runs a few optimizer steps only to verify backward pass. Skip it if you are confident and want to go straight to full training.

In [8]:
RUN_SMOKE_TRAIN = False

if RUN_SMOKE_TRAIN:
    !python scripts/train.py       --config configs/e2e_gpt2_medium_lora.yaml       --smoke-train       --device cuda       --dry-run-max-examples 80       --dry-run-batch-size 8       --smoke-max-steps 10

## 9. Full Training With End-Of-Epoch Validation

This uses the closer paper-aligned config: seed `110`, LoRA rank `4`, alpha `32`, dropout `0.1`, AdamW LR `2e-4`, epsilon `1e-6`, no gradient clipping, 5 epochs.

The training script now evaluates validation loss and perplexity after each completed epoch and appends records to `outputs/runs/e2e_lora_r4_alpha32/metrics.jsonl`.

In [9]:
!python scripts/train.py   --config configs/e2e_gpt2_medium_lora.yaml   --train   --device cuda

Loading weights: 100% 292/292 [00:00<00:00, 1116.87it/s, Materializing param=transformer.wte.weight]
GPT2LMHeadModel LOAD REPORT from: gpt2-medium
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
dry_run=False
smoke_train=False
train=True
device=cuda
replaced_modules=24
trainable_parameters=393216
optimizer_param_groups=1
scheduler=LambdaLR
dry_run_dataset=/content/CS4782-final-project/lora-gpt2-medium-e2e/data/processed/e2e_gpt2/train.jsonl
dry_run_examples=2
dry_run_batch_input_shape=(1, 52)
dry_run_batch_label_shape=(1, 52)
train_dataset=/content/CS4782-final-project/lora-gpt2-medium-e2e/data/processed/e2e_gpt2/train.jsonl
train_examples=42061
train_batch_size=8
validation_batch_size=4
configured_training_steps=26290
effective_training_steps=26290
output_dir=/content/CS4782-final-project/lora-g

## 10. Inspect Training And Validation Logs

In [10]:
import json
from pathlib import Path

metrics_path = Path('outputs/runs/e2e_lora_r4_alpha32/metrics.jsonl')
records = [json.loads(line) for line in metrics_path.read_text().splitlines() if line.strip()]
validation = [record for record in records if record.get('type') == 'validation']
print('total metric records:', len(records))
print('validation records:')
for record in validation:
    print(record)

!ls -lh outputs/runs/e2e_lora_r4_alpha32/checkpoints | tail

total metric records: 26295
validation records:
{'epoch': 1, 'step': 5258, 'type': 'validation', 'valid_dataset': '/content/CS4782-final-project/lora-gpt2-medium-e2e/data/processed/e2e_gpt2/valid.jsonl', 'valid_examples': 4672, 'valid_loss': 2.559605550786404, 'valid_ppl': 12.93071579878652}
{'epoch': 2, 'step': 10516, 'type': 'validation', 'valid_dataset': '/content/CS4782-final-project/lora-gpt2-medium-e2e/data/processed/e2e_gpt2/valid.jsonl', 'valid_examples': 4672, 'valid_loss': 2.5076400439624917, 'valid_ppl': 12.27592520507646}
{'epoch': 3, 'step': 15774, 'type': 'validation', 'valid_dataset': '/content/CS4782-final-project/lora-gpt2-medium-e2e/data/processed/e2e_gpt2/valid.jsonl', 'valid_examples': 4672, 'valid_loss': 2.48658567279169, 'valid_ppl': 12.020165200224463}
{'epoch': 4, 'step': 21032, 'type': 'validation', 'valid_dataset': '/content/CS4782-final-project/lora-gpt2-medium-e2e/data/processed/e2e_gpt2/valid.jsonl', 'valid_examples': 4672, 'valid_loss': 2.4720189389708924,

## 11. Back Up Training Outputs To Drive

This backs up the whole local run directory into:

```text
/content/drive/MyDrive/e2e_lora_r4_alpha32/
```

In [11]:
backup_to_run(LOCAL_RUN_DIR, destination_name='.')
backup_to_run('configs/e2e_gpt2_medium_lora.yaml', destination_name='config_source.yaml')
backup_to_run('colab_train_lora.ipynb')

print('Training backup run folder:', DRIVE_RUN_DIR)
!du -sh /content/drive/MyDrive/e2e_lora_r4_alpha32
!find /content/drive/MyDrive/e2e_lora_r4_alpha32 -maxdepth 2 -type f | sort | tail -20

backed up: outputs/runs/e2e_lora_r4_alpha32 -> /content/drive/MyDrive/e2e_lora_r4_alpha32
backed up: configs/e2e_gpt2_medium_lora.yaml -> /content/drive/MyDrive/e2e_lora_r4_alpha32/config_source.yaml
backed up: colab_train_lora.ipynb -> /content/drive/MyDrive/e2e_lora_r4_alpha32/colab_train_lora.ipynb
Training backup run folder: /content/drive/MyDrive/e2e_lora_r4_alpha32
127M	/content/drive/MyDrive/e2e_lora_r4_alpha32
/content/drive/MyDrive/e2e_lora_r4_alpha32/checkpoints/adapter_step_20000.pt
/content/drive/MyDrive/e2e_lora_r4_alpha32/checkpoints/adapter_step_2000.pt
/content/drive/MyDrive/e2e_lora_r4_alpha32/checkpoints/adapter_step_21000.pt
/content/drive/MyDrive/e2e_lora_r4_alpha32/checkpoints/adapter_step_22000.pt
/content/drive/MyDrive/e2e_lora_r4_alpha32/checkpoints/adapter_step_23000.pt
/content/drive/MyDrive/e2e_lora_r4_alpha32/checkpoints/adapter_step_24000.pt
/content/drive/MyDrive/e2e_lora_r4_alpha32/checkpoints/adapter_step_25000.pt
/content/drive/MyDrive/e2e_lora_r4_alpha

## 12. Resume Training From A Checkpoint, If Needed

Only run this if Colab disconnects before training finishes. Set `RESUME_CHECKPOINT` to the latest checkpoint in Drive or local outputs.

In [ ]:
# RUN_RESUME = False
# RESUME_CHECKPOINT = 'outputs/runs/e2e_lora_r4_alpha32/checkpoints/adapter_step_1000.pt'

# if RUN_RESUME:
#     !python scripts/train.py       --config configs/e2e_gpt2_medium_lora.yaml       --train       --device cuda       --resume-checkpoint "$RESUME_CHECKPOINT

## 13. Generate Full Test Predictions

This uses the configured `generation.decoder`, currently `official_beam`, which is closer to Microsoft `gpt2_beam.py` than Hugging Face `generate()`.

`BATCH_SIZE=4` is conservative. If you have a large GPU, try `8`; if you hit OOM, lower it.

In [17]:
BATCH_SIZE = 16

!TOKENIZERS_PARALLELISM=false TRANSFORMERS_VERBOSITY=error python scripts/generate.py   --config configs/e2e_gpt2_medium_lora.yaml   --split test   --adapter "$LOCAL_ADAPTER"   --batch-size "$BATCH_SIZE"

!ls -lh outputs/runs/e2e_lora_r4_alpha32/generations_test.txt
!python - <<'PY'
from pathlib import Path
path = Path('outputs/runs/e2e_lora_r4_alpha32/generations_test.txt')
print('prediction lines:', sum(1 for _ in path.open()))

Loading weights: 100% 292/292 [00:00<00:00, 922.68it/s, Materializing param=transformer.wte.weight]
generating test: 100% 294/294 [12:37<00:00,  2.58s/batch]
wrote 4693 predictions to /content/CS4782-final-project/lora-gpt2-medium-e2e/outputs/runs/e2e_lora_r4_alpha32/generations_test.txt
-rw-r--r-- 1 root root 1.6M Apr 26 00:29 outputs/runs/e2e_lora_r4_alpha32/generations_test.txt
/bin/bash: line 1: warning: here-document at line 1 delimited by end-of-file (wanted `PY')
prediction lines: 4693


## 14a. Run Grouped E2E Metrics

The main `bleu` and `rouge_l` values are grouped by unique meaning representation with multiple references, matching the official E2E evaluation structure more closely. The `line_*` metrics are stricter debugging metrics.

In [18]:
!python scripts/evaluate.py --config configs/e2e_gpt2_medium_lora.yaml
!cat outputs/runs/e2e_lora_r4_alpha32/generations_test.metrics.json

wrote metrics to /content/CS4782-final-project/lora-gpt2-medium-e2e/outputs/runs/e2e_lora_r4_alpha32/generations_test.metrics.json
{
  "bleu": 54.784983808944006,
  "line_bleu": 23.002262572586694,
  "line_rouge_l": 46.24942281341215,
  "num_examples": 4693,
  "num_unique_mrs": 630,
  "official_predictions_file": "/content/CS4782-final-project/lora-gpt2-medium-e2e/outputs/runs/e2e_lora_r4_alpha32/generations_test.e2e_preds.txt",
  "official_references_file": "/content/CS4782-final-project/lora-gpt2-medium-e2e/outputs/runs/e2e_lora_r4_alpha32/generations_test.e2e_refs.txt",
  "rouge_l": 63.93138740169847
}

#14b. Optional: official E2E NLG metrics.

In [19]:
!mkdir -p external
![ -d external/e2e-metrics/.git ] || git clone https://github.com/tuetschek/e2e-metrics.git external/e2e-metrics
REF_FILE="outputs/runs/e2e_lora_r4_alpha32/generations_test.e2e_refs.txt"
PRED_FILE="outputs/runs/e2e_lora_r4_alpha32/generations_test.e2e_preds.txt"
OUT_FILE="outputs/runs/e2e_lora_r4_alpha32/generations_test.official_e2e_metrics.txt"
!python external/e2e-metrics/measure_scores.py "$REF_FILE" "$PRED_FILE" -p 2>&1 | tee "$OUT_FILE"

Cloning into 'external/e2e-metrics'...
remote: Enumerating objects: 909, done.
remote: Counting objects: 100% (2/2), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 909 (delta 0), reused 0 (delta 0), pack-reused 907 (from 2)
Receiving objects: 100% (909/909), 106.78 MiB | 38.18 MiB/s, done.
Resolving deltas: 100% (492/492), done.
/content/CS4782-final-project/lora-gpt2-medium-e2e/external/e2e-metrics/measure_scores.py:208: SyntaxWarning: invalid escape sequence '\.'
  elif re.search('\.[ct]sv$', sys_file, re.I):
/content/CS4782-final-project/lora-gpt2-medium-e2e/external/e2e-metrics/measure_scores.py:216: SyntaxWarning: invalid escape sequence '\.'
  if re.search('\.[ct]sv$', ref_file, re.I):
Running MS-COCO evaluator...
creating index...
index created!
Loading and preparing results...     
DONE (t=0.00s)
creating index...
index created!
tokenization...
PTBTokenizer tokenized 132573 tokens at 599243.92 tokens per second.
PTBTokenizer tokenized 18741 tokens at 186354.

## 15. Create Figures

Figures are written directly inside the run folder at `outputs/runs/e2e_lora_r4_alpha32/figures`, so downloading one run folder includes its plots.

In [20]:
!python scripts/make_figures.py   --config configs/e2e_gpt2_medium_lora.yaml   --figures-dir outputs/runs/e2e_lora_r4_alpha32/figures

!ls -lh outputs/runs/e2e_lora_r4_alpha32/figures
!cat outputs/runs/e2e_lora_r4_alpha32/figures/summary.json

wrote figures to /content/CS4782-final-project/lora-gpt2-medium-e2e/outputs/runs/e2e_lora_r4_alpha32/figures
wrote summary to /content/CS4782-final-project/lora-gpt2-medium-e2e/outputs/runs/e2e_lora_r4_alpha32/figures/summary.json
total 264K
-rw-r--r-- 1 root root  51K Apr 26 00:32 epoch_loss.png
-rw-r--r-- 1 root root  28K Apr 26 00:32 evaluation_metrics.png
-rw-r--r-- 1 root root 1.4K Apr 26 00:32 summary.json
-rw-r--r-- 1 root root  37K Apr 26 00:32 trainable_parameters.png
-rw-r--r-- 1 root root  84K Apr 26 00:32 training_loss.png
-rw-r--r-- 1 root root  54K Apr 26 00:32 validation_loss.png
{
  "epoch_loss": {
    "epoch_1_mean_loss": 2.859274376159659,
    "epoch_2_mean_loss": 2.6489159269176903,
    "epoch_3_mean_loss": 2.608970800567009,
    "epoch_4_mean_loss": 2.587419120156298,
    "epoch_5_mean_loss": 2.574115697037997
  },
  "figures_dir": "/content/CS4782-final-project/lora-gpt2-medium-e2e/outputs/runs/e2e_lora_r4_alpha32/figures",
  "metrics": {
    "bleu": 54.78498380894

## 16. Back Up Evaluation Outputs And Figures To Drive

This copies the final local run folder again, including the `figures/` subfolder created inside the run.

In [21]:
backup_to_run(LOCAL_RUN_DIR, destination_name='.')
backup_to_run('configs/e2e_gpt2_medium_lora.yaml', destination_name='config_source.yaml')
backup_to_run('colab_train_lora.ipynb')

print('Backed up complete run to:', DRIVE_RUN_DIR)
!find /content/drive/MyDrive/e2e_lora_r4_alpha32 -maxdepth 3 -type f | sort | tail -60

backed up: outputs/runs/e2e_lora_r4_alpha32 -> /content/drive/MyDrive/e2e_lora_r4_alpha32
backed up: configs/e2e_gpt2_medium_lora.yaml -> /content/drive/MyDrive/e2e_lora_r4_alpha32/config_source.yaml
backed up: colab_train_lora.ipynb -> /content/drive/MyDrive/e2e_lora_r4_alpha32/colab_train_lora.ipynb
Backed up complete run to: /content/drive/MyDrive/e2e_lora_r4_alpha32
/content/drive/MyDrive/e2e_lora_r4_alpha32/checkpoints/adapter_final.pt
/content/drive/MyDrive/e2e_lora_r4_alpha32/checkpoints/adapter_step_10000.pt
/content/drive/MyDrive/e2e_lora_r4_alpha32/checkpoints/adapter_step_1000.pt
/content/drive/MyDrive/e2e_lora_r4_alpha32/checkpoints/adapter_step_11000.pt
/content/drive/MyDrive/e2e_lora_r4_alpha32/checkpoints/adapter_step_12000.pt
/content/drive/MyDrive/e2e_lora_r4_alpha32/checkpoints/adapter_step_13000.pt
/content/drive/MyDrive/e2e_lora_r4_alpha32/checkpoints/adapter_step_14000.pt
/content/drive/MyDrive/e2e_lora_r4_alpha32/checkpoints/adapter_step_15000.pt
/content/drive/My